In [ ]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable
import uuid

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

catalog = "cdac-project"

# One schema per document type — matches Notebooks 4 and 5, which
# read/write each type's data here too.
DOCUMENT_TYPE_SCHEMAS = {
    "RESUME": "resume",
    "EMAIL": "email",
    "INVOICE": "invoice",
    "BANK_STATEMENT": "bank_statement",
    "PRESCRIPTION": "prescription",
}

for schema_name in DOCUMENT_TYPE_SCHEMAS.values():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema_name}`")

# Input: Notebook 5's per-type validated output.
validated_resume_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['RESUME']}`.validated_resume_data"
validated_email_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['EMAIL']}`.validated_email_data"
validated_invoice_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['INVOICE']}`.validated_invoice_data"
validated_bank_statement_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['BANK_STATEMENT']}`.validated_bank_statement_data"
validated_prescription_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['PRESCRIPTION']}`.validated_prescription_data"

# Output. NOTE: the spec names this table "<type>_structured_data",
# but that name is already taken — Notebook 4 writes its raw
# (unvalidated, non-audited) extraction output to a table with that
# exact name, and Notebook 5 depends on reading it. Reusing the name
# here would either collide with a completely different schema or
# silently overwrite Notebook 4's output, breaking Notebook 5's
# input. Using a distinctly named "*_final_data" table per type for
# this notebook's clean, audited, upserted storage layer instead.
resume_final_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['RESUME']}`.resume_final_data"
email_final_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['EMAIL']}`.email_final_data"
invoice_final_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['INVOICE']}`.invoice_final_data"
bank_statement_final_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['BANK_STATEMENT']}`.bank_statement_final_data"
prescription_final_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['PRESCRIPTION']}`.prescription_final_data"

# Duplicates are logged here rather than silently dropped (see
# section 9) — one duplicates table per type, same reasoning as the
# final tables above.
resume_duplicates_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['RESUME']}`.resume_duplicate_records"
email_duplicates_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['EMAIL']}`.email_duplicate_records"
invoice_duplicates_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['INVOICE']}`.invoice_duplicate_records"
bank_statement_duplicates_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['BANK_STATEMENT']}`.bank_statement_duplicate_records"
prescription_duplicates_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['PRESCRIPTION']}`.prescription_duplicate_records"

# One ID per run of this notebook, stamped on every row it writes —
# lets you trace which pipeline run produced/touched any given
# record, across every document type.
pipeline_run_id = str(uuid.uuid4())

In [ ]:
# ============================================================
# 3. FALLBACK SCHEMAS (validated tables)
# ============================================================
# Used when this notebook runs before Notebook 5 has ever produced a
# row for that document type — spark.table() on a missing table
# raises an unhandled AnalysisException otherwise. Each schema
# matches what Notebook 5 actually writes for that type: its
# structured fields plus every validation column it adds
# (missing_required_fields, the per-field *_validation_status
# columns, validation_errors, validation_status, duplicate_flag,
# extraction_confidence_score, validation_timestamp).

VALIDATED_RESUME_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, name STRING, email STRING, phone STRING,
linkedin STRING, github STRING, skills ARRAY<STRING>, education ARRAY<STRING>,
experience ARRAY<STRING>, projects ARRAY<STRING>, certifications ARRAY<STRING>,
summary STRING, processing_status STRING, error_message STRING,
processing_timestamp TIMESTAMP, missing_required_fields ARRAY<STRING>,
email_validation_status STRING, phone_normalized STRING,
phone_validation_status STRING, linkedin_validation_status STRING,
github_validation_status STRING, validation_errors ARRAY<STRING>,
validation_status STRING, duplicate_flag BOOLEAN,
extraction_confidence_score INT, validation_timestamp TIMESTAMP
"""

VALIDATED_EMAIL_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, sender STRING, recipient STRING,
subject STRING, sent_date STRING, body STRING, processing_status STRING,
error_message STRING, processing_timestamp TIMESTAMP,
missing_required_fields ARRAY<STRING>, sender_validation_status STRING,
recipient_validation_status STRING, validation_errors ARRAY<STRING>,
validation_status STRING, duplicate_flag BOOLEAN,
extraction_confidence_score INT, validation_timestamp TIMESTAMP
"""

VALIDATED_INVOICE_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, invoice_number STRING, invoice_date STRING,
due_date STRING, total_amount STRING, processing_status STRING,
error_message STRING, processing_timestamp TIMESTAMP,
missing_required_fields ARRAY<STRING>, total_amount_validation_status STRING,
validation_errors ARRAY<STRING>, validation_status STRING, duplicate_flag BOOLEAN,
extraction_confidence_score INT, validation_timestamp TIMESTAMP
"""

VALIDATED_BANK_STATEMENT_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, account_number STRING, statement_period STRING,
opening_balance STRING, closing_balance STRING, transactions ARRAY<STRING>,
processing_status STRING, error_message STRING, processing_timestamp TIMESTAMP,
missing_required_fields ARRAY<STRING>, opening_balance_validation_status STRING,
closing_balance_validation_status STRING, validation_errors ARRAY<STRING>,
validation_status STRING, duplicate_flag BOOLEAN,
extraction_confidence_score INT, validation_timestamp TIMESTAMP
"""

VALIDATED_PRESCRIPTION_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, patient_name STRING, physician_name STRING,
prescription_date STRING, diagnosis STRING, medications ARRAY<STRING>,
processing_status STRING, error_message STRING, processing_timestamp TIMESTAMP,
missing_required_fields ARRAY<STRING>, validation_errors ARRAY<STRING>,
validation_status STRING, duplicate_flag BOOLEAN,
extraction_confidence_score INT, validation_timestamp TIMESTAMP
"""

In [ ]:
# ============================================================
# 4. FINAL STORAGE SCHEMAS
# ============================================================
# The final, clean storage schema per type — narrower than its
# validated table on purpose. Validation-specific columns
# (validation_errors, duplicate_flag, missing_required_fields, ...)
# are useful in the validation layer but don't belong in the clean
# storage layer; each table only carries what a downstream consumer
# of "a <type> record" actually needs, plus audit columns.

RESUME_FINAL_SCHEMA = """
file_id STRING,
document_type STRING,
name STRING,
email STRING,
phone STRING,
linkedin STRING,
github STRING,
skills ARRAY<STRING>,
education ARRAY<STRING>,
experience ARRAY<STRING>,
projects ARRAY<STRING>,
certifications ARRAY<STRING>,
summary STRING,
extraction_confidence_score INT,
source_file STRING,
pipeline_run_id STRING,
created_timestamp TIMESTAMP,
updated_timestamp TIMESTAMP
"""

EMAIL_FINAL_SCHEMA = """
file_id STRING,
document_type STRING,
sender STRING,
recipient STRING,
subject STRING,
sent_date STRING,
body STRING,
extraction_confidence_score INT,
source_file STRING,
pipeline_run_id STRING,
created_timestamp TIMESTAMP,
updated_timestamp TIMESTAMP
"""

INVOICE_FINAL_SCHEMA = """
file_id STRING,
document_type STRING,
invoice_number STRING,
invoice_date STRING,
due_date STRING,
total_amount STRING,
extraction_confidence_score INT,
source_file STRING,
pipeline_run_id STRING,
created_timestamp TIMESTAMP,
updated_timestamp TIMESTAMP
"""

BANK_STATEMENT_FINAL_SCHEMA = """
file_id STRING,
document_type STRING,
account_number STRING,
statement_period STRING,
opening_balance STRING,
closing_balance STRING,
transactions ARRAY<STRING>,
extraction_confidence_score INT,
source_file STRING,
pipeline_run_id STRING,
created_timestamp TIMESTAMP,
updated_timestamp TIMESTAMP
"""

PRESCRIPTION_FINAL_SCHEMA = """
file_id STRING,
document_type STRING,
patient_name STRING,
physician_name STRING,
prescription_date STRING,
diagnosis STRING,
medications ARRAY<STRING>,
extraction_confidence_score INT,
source_file STRING,
pipeline_run_id STRING,
created_timestamp TIMESTAMP,
updated_timestamp TIMESTAMP
"""

# The duplicates log doesn't need MERGE semantics (see section 9), so
# a simpler, SHARED schema is enough across every document type —
# just the identifying fields plus a human-readable summary of which
# identity columns matched, and why/when it was flagged.
DOCUMENT_DUPLICATES_SCHEMA = """
file_id STRING,
document_type STRING,
identity_summary STRING,
pipeline_run_id STRING,
logged_timestamp TIMESTAMP
"""

In [ ]:
# ============================================================
# 5. STORAGE CONFIG
# ============================================================
# One entry per document type — this is what turns the generic
# functions in section 7 into resume-specific / email-specific /
# etc. behavior. Adding a 6th document type later means adding one
# entry here (plus its schemas in sections 3-4) — sections 6, 9 and
# 10's loops stay unchanged.
#
# scalar_fields is a list of (output_column, source_column) pairs —
# they're the same name for every field except resume's phone, which
# stores Notebook 5's cleaned phone_normalized under the "phone"
# name in the final table (see section 7's docstring).

STORAGE_CONFIG = {
    "RESUME": {
        "validated_table": validated_resume_table,
        "fallback_schema": VALIDATED_RESUME_FALLBACK_SCHEMA,
        "final_table": resume_final_table,
        "final_schema": RESUME_FINAL_SCHEMA,
        "duplicates_table": resume_duplicates_table,
        "scalar_fields": [("name", "name"), ("email", "email"), ("phone", "phone_normalized"), ("linkedin", "linkedin"), ("github", "github")],
        "array_fields": ["skills", "education", "experience", "projects", "certifications"],
        "identity_fields": ["name", "email"],
    },
    "EMAIL": {
        "validated_table": validated_email_table,
        "fallback_schema": VALIDATED_EMAIL_FALLBACK_SCHEMA,
        "final_table": email_final_table,
        "final_schema": EMAIL_FINAL_SCHEMA,
        "duplicates_table": email_duplicates_table,
        "scalar_fields": [("sender", "sender"), ("recipient", "recipient"), ("subject", "subject"), ("sent_date", "sent_date"), ("body", "body")],
        "array_fields": [],
        "identity_fields": ["sender", "subject"],
    },
    "INVOICE": {
        "validated_table": validated_invoice_table,
        "fallback_schema": VALIDATED_INVOICE_FALLBACK_SCHEMA,
        "final_table": invoice_final_table,
        "final_schema": INVOICE_FINAL_SCHEMA,
        "duplicates_table": invoice_duplicates_table,
        "scalar_fields": [("invoice_number", "invoice_number"), ("invoice_date", "invoice_date"), ("due_date", "due_date"), ("total_amount", "total_amount")],
        "array_fields": [],
        "identity_fields": ["invoice_number"],
    },
    "BANK_STATEMENT": {
        "validated_table": validated_bank_statement_table,
        "fallback_schema": VALIDATED_BANK_STATEMENT_FALLBACK_SCHEMA,
        "final_table": bank_statement_final_table,
        "final_schema": BANK_STATEMENT_FINAL_SCHEMA,
        "duplicates_table": bank_statement_duplicates_table,
        "scalar_fields": [("account_number", "account_number"), ("statement_period", "statement_period"), ("opening_balance", "opening_balance"), ("closing_balance", "closing_balance")],
        "array_fields": ["transactions"],
        "identity_fields": ["account_number", "statement_period"],
    },
    "PRESCRIPTION": {
        "validated_table": validated_prescription_table,
        "fallback_schema": VALIDATED_PRESCRIPTION_FALLBACK_SCHEMA,
        "final_table": prescription_final_table,
        "final_schema": PRESCRIPTION_FINAL_SCHEMA,
        "duplicates_table": prescription_duplicates_table,
        "scalar_fields": [("patient_name", "patient_name"), ("physician_name", "physician_name"), ("prescription_date", "prescription_date"), ("diagnosis", "diagnosis")],
        "array_fields": ["medications"],
        "identity_fields": ["patient_name", "prescription_date"],
    },
}

In [ ]:
# ============================================================
# 6. CREATE DELTA TABLES IF NOT EXISTS
# ============================================================

def create_delta_table(table_name, ddl_schema):
    """
    Creates an empty Delta table with the given schema if it doesn't
    already exist. Safe to call on every run — CREATE TABLE IF NOT
    EXISTS is a no-op when the table is already there.
    """

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {ddl_schema}
        )
        USING DELTA
    """)


for document_type, config in STORAGE_CONFIG.items():
    create_delta_table(config["final_table"], config["final_schema"])
    create_delta_table(config["duplicates_table"], DOCUMENT_DUPLICATES_SCHEMA)

In [ ]:
# ============================================================
# 7. PREPARE DATA FOR STORAGE
# ============================================================

def prepare_storage_dataframe(df, config, run_id):
    """
    Selects and cleans exactly the columns this type's final schema
    needs, trims stray whitespace from every string field (including
    inside array fields — a "  Amoxicillin " entry in medications is
    just as unclean as an untrimmed name), and stamps every row with
    this run's audit metadata. All column-level, no collect() / no
    row-by-row Python.

    config["scalar_fields"] is (output_column, source_column) pairs
    rather than a plain list of names because resume is the one type
    where the final column isn't a direct copy: "phone" in the final
    table comes from Notebook 5's phone_normalized (already
    digits-only with the country code stripped), not the raw phone
    field — exactly what "data cleaning before storage" is asking
    for. Every other type's output_column and source_column are the
    same name.
    """

    columns = [F.col("file_id"), F.col("document_type")]

    for output_column, source_column in config["scalar_fields"]:
        columns.append(F.trim(F.col(source_column)).alias(output_column))

    for field_name in config["array_fields"]:
        columns.append(
            F.coalesce(
                F.transform(F.col(field_name), lambda x: F.trim(x)),
                F.array().cast("array<string>")
            ).alias(field_name)
        )

    columns.append(F.col("extraction_confidence_score"))

    prepared_df = df.select(*columns)

    # source_file: ideally the original file name, but Notebook 4's
    # output schema doesn't carry file_name through from Notebook 3
    # — file_id is the closest per-file identifier actually
    # available here. Fixing this properly means adding file_name to
    # each *_structured_data table upstream in Notebook 4.
    prepared_df = prepared_df.withColumn("source_file", F.col("file_id"))
    prepared_df = prepared_df.withColumn("pipeline_run_id", F.lit(run_id))
    prepared_df = prepared_df.withColumn("created_timestamp", F.current_timestamp())
    prepared_df = prepared_df.withColumn("updated_timestamp", F.current_timestamp())

    # Safety net: Delta's MERGE raises a hard error ("the ON search
    # condition matched multiple rows") if the source has more than
    # one row for the same file_id — which would fail the ENTIRE
    # write, not just one bad record. That shouldn't happen given how
    # upstream incremental processing works, but this is cheap
    # insurance against it — one bad batch failing quietly here beats
    # the whole MERGE blowing up in section 9.
    prepared_df = prepared_df.dropDuplicates(["file_id"])

    return prepared_df

In [ ]:
# ============================================================
# 8. IMPLEMENT MERGE / UPSERT LOGIC
# ============================================================

def merge_into_delta_table(table_name, source_df, merge_key="file_id"):
    """
    Upserts source_df into table_name on merge_key: existing
    file_ids get UPDATEd (their fields refreshed to the latest
    extraction, updated_timestamp bumped, but created_timestamp left
    untouched so it still reflects when the record first entered the
    system) — new file_ids get INSERTed. Never overwrites the whole
    table. Used for every document type — the merge logic itself
    doesn't care what kind of document the rows represent.
    """

    delta_table = DeltaTable.forName(spark, table_name)

    update_columns = [c for c in source_df.columns if c != "created_timestamp"]
    update_set = {c: f"source.{c}" for c in update_columns}
    insert_set = {c: f"source.{c}" for c in source_df.columns}

    (
        delta_table.alias("target")
        .merge(source_df.alias("source"), f"target.{merge_key} = source.{merge_key}")
        .whenMatchedUpdate(set=update_set)
        .whenNotMatchedInsert(values=insert_set)
        .execute()
    )

In [ ]:
# ============================================================
# 9. RUN STORAGE PIPELINE FOR EVERY DOCUMENT TYPE
# ============================================================
# store_document_type() runs the full pipeline for ONE document
# type: read validated data (with fallback + incremental skip of
# file_ids already stored), keep only PASS records, split off
# duplicates (logged, not silently dropped — see
# DOCUMENT_DUPLICATES_SCHEMA in section 4), then MERGE the rest into
# that type's final table. The loop at the bottom runs it once per
# type. Same shape as Notebooks 4 and 5's registry/config pattern.

def store_document_type(document_type, config, run_id):
    validated_table = config["validated_table"]
    final_table = config["final_table"]
    duplicates_table = config["duplicates_table"]

    if spark.catalog.tableExists(validated_table):
        validated_df = spark.table(validated_table)
    else:
        validated_df = spark.createDataFrame([], schema=config["fallback_schema"])

    # Incremental — skip file_ids already stored in this type's final
    # table, so a record that's already correctly stored doesn't get
    # needlessly re-merged (and its updated_timestamp/pipeline_run_id
    # bumped for no real reason) every time this notebook runs. Safe
    # to filter this early — nothing downstream needs cross-row
    # context beyond what Notebook 5 already computed per row.
    if spark.catalog.tableExists(final_table):
        already_stored_ids = {
            row.file_id
            for row in spark.table(final_table).select("file_id").collect()
        }
        validated_df = validated_df.filter(~F.col("file_id").isin(already_stored_ids))

    # NOTE: the spec says to filter where validation_status =
    # "SUCCESS", but Notebook 5 actually writes "PASS" / "FAILED" —
    # using the real value so this notebook actually matches anything.
    valid_df = validated_df.filter(F.col("validation_status") == "PASS")

    # Records Notebook 5 flagged as duplicates don't go into the
    # clean final table at all — but they're not silently discarded
    # either; logging them means you can go back and see what got
    # excluded and why, instead of just missing data with no trail.
    non_duplicate_df = valid_df.filter(F.col("duplicate_flag") == False)
    duplicate_df = valid_df.filter(F.col("duplicate_flag") == True)

    identity_parts = [
        F.concat(F.lit(f"{field}: "), F.coalesce(F.col(field), F.lit("")))
        for field in config["identity_fields"]
    ]

    duplicate_log_df = (
        duplicate_df
        .withColumn("identity_summary", F.concat_ws(" | ", *identity_parts))
        .select("file_id", "document_type", "identity_summary")
        .withColumn("pipeline_run_id", F.lit(run_id))
        .withColumn("logged_timestamp", F.current_timestamp())
    )

    duplicate_log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(duplicates_table)

    storage_ready_df = prepare_storage_dataframe(non_duplicate_df, config, run_id)
    merge_into_delta_table(final_table, storage_ready_df, merge_key="file_id")


for document_type, config in STORAGE_CONFIG.items():
    store_document_type(document_type, config, pipeline_run_id)
    print(f"Stored {document_type}")

In [ ]:
# ============================================================
# 10. VERIFY STORED RECORDS
# ============================================================

def validate_storage_output(table_name):
    """
    Post-write sanity check for one final table. Uses
    count()/aggregates only — these are single-number results pulled
    to the driver, not the thousands-to-millions of underlying rows,
    so this stays cheap regardless of table size.
    """

    stored_df = spark.table(table_name)

    total_count = stored_df.count()
    this_run_count = stored_df.filter(F.col("pipeline_run_id") == pipeline_run_id).count()

    print(f"Total records in {table_name}: {total_count}")
    print(f"Records inserted/updated by this run ({pipeline_run_id}): {this_run_count}")

    return total_count


for document_type, config in STORAGE_CONFIG.items():
    validate_storage_output(config["final_table"])

In [ ]:
# ============================================================
# 11. DISPLAY FINAL OUTPUT
# ============================================================

for document_type, config in STORAGE_CONFIG.items():
    print(f"--- {document_type} ---")
    display(spark.table(config["final_table"]))